# Entropy-SDE with Rényi Divergence
### A Study of Divergence Choice in Diffusion-based Low-Light Image Enhancement

**Base method:** *Equipping Diffusion Models with Differentiable Spatial Entropy for Low-Light
Image Enhancement* (Lian et al., CVPRW 2024, NTIRE 2024 — arXiv:2404.09735; official code:
`shermanlian/spatial-entropy-loss`). The method replaces the pointwise L2 noise-matching
objective in diffusion training with a **KL divergence** between KDE-estimated spatial
distributions of the true vs. predicted noise.

**This notebook's contribution:** we generalize that spatial loss from KL divergence to the
**Rényi divergence family** `D_α(P‖Q)`, of which KL is the special case `α → 1`. We derive the
relevant properties of `D_α` (Section 7), empirically compare KL against ten `α` values spanning
`{0.5, ..., 3.0}` on real images (Section 9), then select a single best-performing `α` via a
**two-stage ablation scored by held-out validation PSNR/SSIM** (Section 15) — not raw training
loss, since `D_α`'s own monotonicity in `α` makes raw loss values incomparable across candidates —
and use *that one value* for the full training run (Section 17), mirroring how a hyperparameter
would be chosen and reported in a paper. The spatial KDE itself (Section 6) uses true spatial
neighbors averaged over 4 offsets, not a random pixel pairing, so the divergence is actually
measuring local texture statistics rather than global intensity co-occurrence.

**Dataset:** real photographs, with a proper **train / validation / test split** (Section 3) so
that the split used to pick `α` is never the split used to report a final number. We attempt to
auto-download the **official LOL-v1** dataset (485 real low/normal-light training pairs + 15 test
pairs, Wei et al., BMVC 2018); if that fails (no internet, mirror down) we fall back to real photos
bundled with `scikit-image`, synthetically darkened. Either way, every image in this notebook is a
real photograph, not synthetic noise.

**Contents:** Setup → Config → Dataset (real download + train/val/test split) → Model → Diffusion
→ Spatial KDE (math, true-neighbor fix) → Divergence theory (KL & Rényi, math) → Direct
KL-vs-Rényi comparison → Training objective → Optimizer/EMA → Training loop → Sampling → Metrics →
Two-stage α-selection ablation (validation-scored) → KDE cost benchmark → Full training run →
Training curves → KDE heatmaps → Qualitative results → Quantitative test-set summary →
Discussion & limitations → References.


## 1. Setup & Installation
Core dependencies: PyTorch (model + training), OpenCV/scikit-image (image I/O, real sample
photos, PSNR/SSIM), and matplotlib (all plots in this notebook).

In [ ]:
# !pip install torch torchvision numpy opencv-python scikit-image matplotlib tqdm --quiet
import os, math, random, glob, time, urllib.request, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


## 2. Hyperparameters / Config
`patch_size=128` and the diffusion/optimizer settings match the paper's Table 4. `total_iters`
defaults to the paper's `250_000`; the main training run in Section 17 uses the full 250,000-iteration setting.
`renyi_alpha` is left at a placeholder here — Section 15's ablation overwrites it with the
empirically best value before the real training run.

In [ ]:
class Config:
    # --- data / patches ---
    patch_size      = 128          # paper: 128x128
    batch_size      = 16           # paper: 16
    num_workers     = 4

    # --- optimization ---
    lr              = 1e-4         # paper: 1e-4
    weight_decay    = 1e-2
    total_iters     = 250_000      # paper's full-scale setting
    warmup_iters    = 200
    grad_clip       = 1.0
    ema_decay       = 0.999
    mixed_precision = True

    # --- diffusion ---
    diffusion_steps = 100          # paper: 100
    beta_schedule   = 'cosine'

    # --- differentiable spatial entropy / KDE ---
    kde_bins        = 32
    kde_bandwidth   = 0.15
    spatial_shifts  = [(1, 0), (-1, 0), (0, 1), (0, -1)]   # true spatial neighbors, averaged
    use_shuffled_neighbor = False   # NOTE: True destroys the "spatial" relationship entirely --
                                     # it pairs each pixel with a RANDOM pixel from the same image,
                                     # not an actual neighbor. Kept as an option for comparison
                                     # only (see Section 6); default is real spatial neighbors.

    # --- Rényi / KL loss ---
    renyi_alpha     = 1.5          # placeholder; Section 15 sets this to the ablation winner
    divergence      = 'renyi'      # 'renyi' or 'kl'
    lambda_spatial  = 0.5
    lambda_pixel    = 1.0
    loss_mode       = 'combined'   # 'combined' or 'replace'

    # --- model ---
    base_channels   = 32
    channel_mults   = (1, 2, 4)
    time_emb_dim    = 128

cfg = Config()
ALPHA_CANDIDATES = [0.5, 0.75, 0.9, 0.95, 1.0, 1.1, 1.25, 1.5, 2.0, 3.0]   # dense around alpha=1
                    # so the screening in Section 15 can reveal a true optimum away from KL
                    # (alpha=1), not just confirm the coarse trend Section 9 shows.


## 3. Dataset
**Is the paper's real dataset available?** Yes, partially:
- **LOL-v1** (Wei et al., BMVC 2018 — 485 train / 15 test real low/normal-light pairs) has a
  working direct-download mirror (used by Keras' own official Zero-DCE tutorial), so we download
  and use it automatically below.
- **LOL-v2** (real + synthetic subsets, Yang et al. 2021) is **not** available via a simple direct
  link — the official distribution is Google Drive / Baidu Pan only, which can't be fetched
  programmatically without manual credentials/IDs. `LOLDataset(low_dir=..., high_dir=...)` below
  works unchanged if you download LOL-v2 by hand and point it at the extracted folder.

If the LOL-v1 download fails for any reason (offline environment, mirror down), we fall back to
real `scikit-image` sample photographs (synthetically darkened) so the notebook still runs on
genuine images end-to-end.

**Train / validation / test split.** Evaluating on the same images used for training doesn't tell
you anything about generalization, and using the official test set to *choose* a hyperparameter
(like α) leaks test information into a decision that should be made blind to it. So we split
three ways:
- **train**: used to fit model weights (LOL-v1: `our485` minus the last 50 pairs).
- **validation**: used only to *pick* α in Section 15's ablation, never to train on (LOL-v1: the
  held-out last 50 pairs of `our485`).
- **test**: touched exactly once, at the very end (Section 21), to report a final number for
  whichever α validation selected (LOL-v1: the official `eval15` folder).

The `RealSampleDataset` fallback has only 4 base images, so the same three-way split isn't
meaningful there — we hold out 1 of the 4 images as both validation and test and say so plainly
in the printed dataset summary, rather than pretend it's a real split.

In [ ]:
import cv2
from skimage import data as skdata

LOL_V1_URL = "https://huggingface.co/datasets/geekyrakshit/LoL-Dataset/resolve/main/lol_dataset.zip"


class LOLDataset(Dataset):
    """Paired low-light / normal-light dataset (LOL-v1 / LOL-v2 / NTIRE folder layout:
    a `low/` and a `high/` folder with matching filenames). `indices` restricts to a subset of
    files (used to carve train/val splits out of the same folder without touching disk)."""

    def __init__(self, low_dir, high_dir, patch_size=128, train=True, indices=None):
        low_paths = sorted(glob.glob(os.path.join(low_dir, '*.*')))
        high_paths = sorted(glob.glob(os.path.join(high_dir, '*.*')))
        assert len(low_paths) == len(high_paths), 'low/high image count mismatch'
        assert len(low_paths) > 0, f'no images found in {low_dir}'
        if indices is not None:
            low_paths = [low_paths[i] for i in indices]
            high_paths = [high_paths[i] for i in indices]
        self.low_paths = low_paths
        self.high_paths = high_paths
        self.patch_size = patch_size
        self.train = train   # controls random-crop augmentation; False = deterministic center view

    def __len__(self):
        return len(self.low_paths)

    def _read(self, path):
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return img

    def __getitem__(self, idx):
        low = self._read(self.low_paths[idx])
        high = self._read(self.high_paths[idx])
        h, w, _ = low.shape
        ps = self.patch_size

        if self.train:
            if h > ps and w > ps:
                top = random.randint(0, h - ps)
                left = random.randint(0, w - ps)
                low = low[top:top + ps, left:left + ps]
                high = high[top:top + ps, left:left + ps]
            else:
                low = cv2.resize(low, (ps, ps))
                high = cv2.resize(high, (ps, ps))
            if random.random() < 0.5:
                low, high = np.fliplr(low).copy(), np.fliplr(high).copy()
            if random.random() < 0.5:
                low, high = np.flipud(low).copy(), np.flipud(high).copy()
            k = random.randint(0, 3)
            low, high = np.rot90(low, k).copy(), np.rot90(high, k).copy()
        else:
            # deterministic center crop/resize for consistent, comparable validation/test scoring
            if h >= ps and w >= ps:
                top, left = (h - ps) // 2, (w - ps) // 2
                low = low[top:top + ps, left:left + ps]
                high = high[top:top + ps, left:left + ps]
            else:
                low = cv2.resize(low, (ps, ps))
                high = cv2.resize(high, (ps, ps))

        low = torch.from_numpy(low).permute(2, 0, 1) * 2 - 1
        high = torch.from_numpy(high).permute(2, 0, 1) * 2 - 1
        return low.float(), high.float()


def synthesize_low_light(img, gamma=2.6, darken=0.18, noise_std=0.02):
    """Gamma darkening + global brightness cut + sensor noise -> plausible low-light version
    of a real normal-light image. img: float32 HWC in [0,1]."""
    dark = np.power(img, gamma) * darken
    noise = np.random.randn(*dark.shape).astype(np.float32) * noise_std
    return np.clip(dark + noise, 0, 1).astype(np.float32)


class RealSampleDataset(Dataset):
    """Fallback dataset: real scikit-image sample photos, synthetically darkened. No download.
    `split` selects which base image(s) this instance draws patches from, so train/val/test don't
    share source images even in this tiny 4-image fallback."""

    ALL_IMAGE_FNS = {'astronaut': skdata.astronaut, 'chelsea': skdata.chelsea,
                      'coffee': skdata.coffee, 'rocket': skdata.rocket}
    TRAIN_NAMES = ['astronaut', 'chelsea', 'coffee']
    VAL_NAMES = ['rocket']       # held out from training entirely
    TEST_NAMES = ['rocket']      # NOTE: with only 4 images, val and test share the 1 held-out
                                  # image here -- a real limitation of this fallback, stated plainly
                                  # rather than hidden (see Section 22).

    def __init__(self, patch_size=128, length=256, train=True, split='train'):
        self.patch_size = patch_size
        self.length = length
        self.train = train
        names = {'train': self.TRAIN_NAMES, 'val': self.VAL_NAMES, 'test': self.TEST_NAMES}[split]
        self.base_images = [self.ALL_IMAGE_FNS[n]().astype(np.float32) / 255.0 for n in names]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        img = self.base_images[idx % len(self.base_images)]
        h, w, _ = img.shape
        ps = self.patch_size
        if self.train:
            top = random.randint(0, max(h - ps, 0))
            left = random.randint(0, max(w - ps, 0))
        else:
            top, left = max(h - ps, 0) // 2, max(w - ps, 0) // 2
        high = img[top:top + ps, left:left + ps]
        if high.shape[0] != ps or high.shape[1] != ps:
            high = cv2.resize(high, (ps, ps))
        if self.train:
            if random.random() < 0.5:
                high = np.fliplr(high).copy()
            if random.random() < 0.5:
                high = np.flipud(high).copy()
            k = random.randint(0, 3)
            high = np.rot90(high, k).copy()
        low = synthesize_low_light(high)
        low_t = torch.from_numpy(low).permute(2, 0, 1) * 2 - 1
        high_t = torch.from_numpy(high).permute(2, 0, 1) * 2 - 1
        return low_t.float(), high_t.float()


def try_download_lol_v1(dest='lol_dataset', timeout=60):
    """Attempt to fetch + extract the real LOL-v1 dataset. Returns the root dir on success,
    or None on any failure (caller should fall back to RealSampleDataset)."""
    if os.path.isdir(dest) and os.path.isdir(os.path.join(dest, 'our485')):
        return dest
    try:
        zip_path = 'lol_dataset.zip'
        print(f'Attempting to download real LOL-v1 dataset ({LOL_V1_URL}) ...')
        urllib.request.urlretrieve(LOL_V1_URL, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('.')
        os.remove(zip_path)
        print('LOL-v1 downloaded and extracted successfully.')
        return dest
    except Exception as e:
        print(f'Could not download LOL-v1 ({e!r}). Falling back to real sample photographs.')
        return None


VAL_SIZE = 50   # pairs carved out of LOL-v1's our485 for validation (never trained on)

# --- Resolve which real dataset we actually have available, and build train/val/test splits ---
_lol_root = try_download_lol_v1()
_low_dir = _high_dir = _eval_low_dir = _eval_high_dir = None
USING_LOL_V1 = False

if _lol_root is not None:
    _low_dir = os.path.join(_lol_root, 'our485', 'low')
    _high_dir = os.path.join(_lol_root, 'our485', 'high')
    _eval_low_dir = os.path.join(_lol_root, 'eval15', 'low')
    _eval_high_dir = os.path.join(_lol_root, 'eval15', 'high')
    USING_LOL_V1 = os.path.isdir(_low_dir) and os.path.isdir(_high_dir)

if USING_LOL_V1:
    n_total = len(glob.glob(os.path.join(_low_dir, '*.*')))
    train_idx = list(range(0, n_total - VAL_SIZE))
    val_idx = list(range(n_total - VAL_SIZE, n_total))

    real_train_ds = LOLDataset(_low_dir, _high_dir, patch_size=cfg.patch_size, train=True, indices=train_idx)
    real_val_ds = LOLDataset(_low_dir, _high_dir, patch_size=cfg.patch_size, train=False, indices=val_idx)
    real_test_ds = LOLDataset(_eval_low_dir, _eval_high_dir, patch_size=cfg.patch_size, train=False)
    DATASET_SOURCE = (f'LOL-v1 real dataset ({len(real_train_ds)} train / {len(real_val_ds)} val '
                       f'/ {len(real_test_ds)} test real captured pairs)')
else:
    real_train_ds = RealSampleDataset(patch_size=cfg.patch_size, length=256, train=True, split='train')
    real_val_ds = RealSampleDataset(patch_size=cfg.patch_size, length=32, train=False, split='val')
    real_test_ds = RealSampleDataset(patch_size=cfg.patch_size, length=32, train=False, split='test')
    DATASET_SOURCE = ('RealSampleDataset fallback (3 images train / 1 image val+test -- val and '
                       'test SHARE their one held-out image here, an explicit limitation of this '
                       'tiny fallback; see Section 22)')

# real_ds kept as an alias to the training split for backward-compatible use in Sections 6-9,
# which only need *some* real images to demonstrate the math, not a rigorous split.
real_ds = real_train_ds
print(f'Dataset in use: {DATASET_SOURCE}')


## 4. Model — Conditional NAFNet-inspired Noise Predictor
We use a lightweight **NAFNet-inspired** conditional encoder-decoder (Chen et al., ECCV 2022 —
Nonlinear Activation Free Network), matching the official repo's choice of backbone family.
This is *not* a reproduction of the exact NAFNet architecture (channel counts, block depth, and a
few block-internal details differ) — it borrows NAFNet's core ingredients (`LayerNorm2d`,
`SimpleGate` instead of ReLU/GELU, simplified channel attention) in a smaller network sized for
this notebook's compute budget. The network is conditioned on the low-light image
(channel-concatenated) and the diffusion timestep (sinusoidal embedding injected via per-block
FiLM scale/shift).

In [ ]:
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb


class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = x.var(1, keepdim=True, unbiased=False)
        x = (x - mu) / torch.sqrt(var + self.eps)
        return x * self.weight[None, :, None, None] + self.bias[None, :, None, None]


class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class NAFBlock(nn.Module):
    def __init__(self, channels, time_emb_dim, expand=2):
        super().__init__()
        hidden = channels * expand
        self.norm1 = LayerNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, hidden, 1)
        self.dwconv = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden)
        self.gate1 = SimpleGate()
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(hidden // 2, hidden // 2, 1),
        )
        self.conv2 = nn.Conv2d(hidden // 2, channels, 1)

        self.norm2 = LayerNorm2d(channels)
        self.conv3 = nn.Conv2d(channels, hidden, 1)
        self.gate2 = SimpleGate()
        self.conv4 = nn.Conv2d(hidden // 2, channels, 1)

        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, channels * 2))
        self.gamma = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x, t_emb):
        scale_shift = self.time_mlp(t_emb)[:, :, None, None]
        scale, shift = scale_shift.chunk(2, dim=1)

        y = self.norm1(x)
        y = y * (1 + scale) + shift
        y = self.conv1(y)
        y = self.dwconv(y)
        y = self.gate1(y)
        y = y * self.sca(y)
        y = self.conv2(y)
        x = x + y * self.gamma

        y = self.norm2(x)
        y = self.conv3(y)
        y = self.gate2(y)
        y = self.conv4(y)
        x = x + y * self.beta
        return x


class ConditionalNAFNet(nn.Module):
    def __init__(self, in_ch=3, cond_ch=3, base_ch=32, mults=(1, 2, 4), time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 4, time_emb_dim),
        )
        self.stem = nn.Conv2d(in_ch + cond_ch, base_ch, 3, padding=1)

        chs = [base_ch * m for m in mults]
        self.downs = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        prev_ch = base_ch
        for ch in chs:
            self.downs.append(NAFBlock(prev_ch, time_emb_dim))
            self.down_samples.append(nn.Conv2d(prev_ch, ch, 2, stride=2))
            prev_ch = ch

        self.mid = NAFBlock(prev_ch, time_emb_dim)

        self.ups = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        for ch in reversed(chs[:-1]):
            self.up_samples.append(nn.ConvTranspose2d(prev_ch, ch, 2, stride=2))
            self.ups.append(NAFBlock(ch * 2, time_emb_dim))
            prev_ch = ch * 2
        self.up_samples.append(nn.ConvTranspose2d(prev_ch, base_ch, 2, stride=2))
        self.ups.append(NAFBlock(base_ch * 2, time_emb_dim))

        self.head = nn.Conv2d(base_ch * 2, 3, 3, padding=1)

    def forward(self, x_noisy, cond, t):
        t_emb = self.time_mlp(t)
        h = self.stem(torch.cat([x_noisy, cond], dim=1))

        skips = []
        for block, down in zip(self.downs, self.down_samples):
            h = block(h, t_emb)
            skips.append(h)
            h = down(h)

        h = self.mid(h, t_emb)

        for block, up, skip in zip(self.ups, self.up_samples, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = block(h, t_emb)

        return self.head(h)


## 5. Diffusion Process
Standard conditional DDPM: forward process adds Gaussian noise to the normal-light image; the
network predicts that noise conditioned on the low-light image, matching Section 2 of the paper
(`x0 -> xt -> ε̂θ(xt, t)`).

In [ ]:
def cosine_beta_schedule(steps, s=0.008):
    x = torch.linspace(0, steps, steps + 1)
    alphas_cumprod = torch.cos(((x / steps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.999)


class GaussianDiffusion:
    def __init__(self, steps, schedule='cosine', device='cpu'):
        betas = cosine_beta_schedule(steps) if schedule == 'cosine' else torch.linspace(1e-4, 2e-2, steps)
        self.betas = betas.to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_acp = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_acp = torch.sqrt(1.0 - self.alphas_cumprod)
        self.steps = steps

    def _extract(self, a, t, shape):
        out = a.gather(0, t)
        return out.reshape(-1, *((1,) * (len(shape) - 1)))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_acp_t = self._extract(self.sqrt_acp, t, x0.shape)
        sqrt_om_acp_t = self._extract(self.sqrt_one_minus_acp, t, x0.shape)
        return sqrt_acp_t * x0 + sqrt_om_acp_t * noise, noise

    @torch.no_grad()
    def p_sample_loop(self, model, cond, shape, device):
        img = torch.randn(shape, device=device)
        for i in reversed(range(self.steps)):
            t = torch.full((shape[0],), i, device=device, dtype=torch.long)
            eps_hat = model(img, cond, t)
            alpha_t = self.alphas[i]
            acp_t = self.alphas_cumprod[i]
            beta_t = self.betas[i]
            coef = beta_t / torch.sqrt(1 - acp_t)
            mean = (1 / torch.sqrt(alpha_t)) * (img - coef * eps_hat)
            if i > 0:
                noise = torch.randn_like(img)
                img = mean + torch.sqrt(beta_t) * noise
            else:
                img = mean
        return img.clamp(-1, 1)


## 6. Differentiable Spatial KDE

**Idea.** For an image, look at pairs `(i, j)` of intensities: a pixel's own value `i` and a
neighboring pixel's value `j`. The joint histogram of all such pairs across the image is a
2-D empirical distribution `P(i, j)` that captures *local texture statistics* — smooth regions
concentrate mass near the diagonal `i≈j`; noisy/textured regions spread mass off-diagonal.

**Why KDE, not a hard histogram.** A hard histogram (counting which bin each pixel falls into) has
zero gradient almost everywhere — useless for backprop. Kernel Density Estimation replaces the
hard "which bin" assignment with a smooth Gaussian weight over *all* bins:

$$w_k(x) = \exp\!\left(-\tfrac{1}{2}\left(\tfrac{x - c_k}{h}\right)^2\right), \quad k = 1,\dots,\text{bins}$$

where `c_k` are bin centers and `h` is the bandwidth. The joint soft histogram is then

$$P(i,j) \;\propto\; \sum_{p \in \text{pixels}} w_i(x_p)\, w_j(\tilde{x}_p)$$

with `x_p` the pixel's value and `x̃_p` its neighbor's value — an outer product of soft
assignments, summed over all pixels and normalized to sum to 1. Every operation here (exp, sum,
normalize) is differentiable, so gradients flow from the divergence loss all the way back through
`P(i,j)` into the network's predicted noise. This directly implements Eq. (9)–(14) of the paper.

**A correctness note on `x̃_p` (the "neighbor").** An earlier version of this notebook set
`use_shuffled_neighbor=True` by default, which pairs each pixel with a *uniformly random* pixel
from the same image rather than an actual spatial neighbor. That silently breaks the "spatial" in
spatial KDE: the resulting `P(i,j)` becomes closer to the outer product of the image's own global
intensity histogram with itself (via random pairing), not a measure of local joint statistics. The
default here is now **true spatial neighbors** — specifically, we average the divergence over four
one-pixel offsets `(±1,0), (0,±1)` rather than a single fixed direction, which is more robust to
the image's dominant edge orientation than any one offset alone. `use_shuffled_neighbor=True`
remains available purely as a comparison point, not the default.

In [ ]:
def spatial_kde(img, bins=cfg.kde_bins, bandwidth=cfg.kde_bandwidth, shift=(1, 0), shuffled=False):
    """Differentiable spatial 2D probability map P(i, j) via Gaussian KDE, using ONE neighbor
    offset (or a random pairing if shuffled=True). img: (B, C, H, W) in [-1, 1].
    Returns (B, C, bins, bins), sums to 1 over (bins,bins). See spatial_kde_multi_offset below for
    the averaged-over-offsets version actually used in training."""
    B, C, H, W = img.shape
    x = img.reshape(B, C, H * W)

    if shuffled:
        perm = torch.randperm(H * W, device=img.device)
        x_neighbor = x[:, :, perm]
    else:
        dy, dx = shift
        neighbor_img = torch.roll(img, shifts=(dy, dx), dims=(2, 3))
        x_neighbor = neighbor_img.reshape(B, C, H * W)

    bin_centers = torch.linspace(-1, 1, bins, device=img.device)

    def soft_hist_weights(vals):
        diff = vals.unsqueeze(-1) - bin_centers.view(1, 1, 1, bins)
        return torch.exp(-0.5 * (diff / bandwidth) ** 2)

    w_i = soft_hist_weights(x)
    w_j = soft_hist_weights(x_neighbor)

    joint = torch.einsum('bcni,bcnj->bcij', w_i, w_j)
    joint = joint / (joint.sum(dim=(-2, -1), keepdim=True) + 1e-12)
    return joint


### 6.1 Visualizing a Spatial Distribution
Before diving into divergences, let's actually *see* what `spatial_kde` produces on a real image
patch: mass concentrated near the diagonal means the image is locally smooth; spread mass means
more local texture/noise. This is the object every divergence in this notebook compares.

In [ ]:
demo_patch_low, demo_patch_high = real_ds[0]
P_demo = spatial_kde(demo_patch_high.unsqueeze(0))[0, 0].detach().numpy()   # first channel

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(((demo_patch_high.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1))
axes[0].set_title('Real image patch (ground truth)')
axes[0].axis('off')

im = axes[1].imshow(P_demo, origin='lower', cmap='inferno')
axes[1].set_title('Spatial KDE distribution P(i, j)\n(red channel, true spatial neighbor)')
axes[1].set_xlabel('neighbor intensity bin j'); axes[1].set_ylabel('pixel intensity bin i')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()


### 6.2 Shuffled vs. True Spatial Neighbor: Seeing the Difference
A direct visual check of the claim above: shuffling destroys spatial structure, so its `P(i,j)`
should look different from the true-neighbor version — closer to what you'd get from two
independent copies of the image's global histogram (mass spread more evenly, less diagonal
concentration for a locally-smooth image).

In [ ]:
P_true_neighbor = spatial_kde(demo_patch_high.unsqueeze(0), shift=(1, 0), shuffled=False)[0, 0].detach().numpy()
P_shuffled = spatial_kde(demo_patch_high.unsqueeze(0), shuffled=True)[0, 0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(P_true_neighbor, origin='lower', cmap='inferno')
axes[0].set_title('True spatial neighbor (default)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(P_shuffled, origin='lower', cmap='inferno')
axes[1].set_title('Shuffled/random pairing\n(legacy option, NOT default)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel('neighbor bin j'); ax.set_ylabel('pixel bin i')
plt.tight_layout()
plt.show()


## 7. Divergence Losses — Mathematical Foundations

**KL divergence** (the paper's original spatial loss) between two discrete distributions `P, Q`
on the same support:

$$D_{KL}(P\|Q) = \sum_i P(i)\log\frac{P(i)}{Q(i)}$$

**Rényi divergence** of order `α` (this notebook's generalization), for `α ∈ (0,1)∪(1,∞)`:

$$D_\alpha(P\|Q) = \frac{1}{\alpha - 1}\log \sum_i P(i)^\alpha Q(i)^{1-\alpha}$$

**Key mathematical properties** (Van Erven & Harremoës, *"Rényi Divergence and Kullback–Leibler
Divergence,"* IEEE Trans. Information Theory, 2014):

1. **Non-negativity:** `D_α(P‖Q) ≥ 0`, with equality iff `P = Q`.
2. **Limit α → 1 recovers KL exactly.** Writing `D_α` as `log(Σ P^α Q^{1-α}) / (α-1)` and applying
   L'Hôpital's rule as `α→1` (both numerator and denominator → 0) gives
   `lim_{α→1} D_α(P‖Q) = Σ P(i) log(P(i)/Q(i)) = D_KL(P‖Q)`. KL is not a separate formula here —
   it is literally the `α=1` point of the same curve.
3. **Monotonicity in α:** for fixed `P, Q`, `D_α(P‖Q)` is non-decreasing in `α`. Consequently
   `α<1` gives a *smaller* (more forgiving, mass-covering) divergence value than KL, and `α>1`
   gives a *larger* (stricter, mode-seeking) value — this is exactly the trend you'll see in
   Section 9's numbers.
4. **Named special cases:**
   - `α = 1/2`: `D_{1/2}(P‖Q) = -2 log Σ√(P(i)Q(i)) = -2 log BC(P,Q)`, the (negative log)
     **Bhattacharyya coefficient** — a symmetric similarity measure.
   - `α = 2`: `D_2(P‖Q) = log(1 + χ²(P‖Q))` where `χ²(P‖Q) = Σ (P(i)-Q(i))²/Q(i)` — directly tied
     to the **Pearson chi-squared divergence**.
   - `α → ∞`: `D_∞(P‖Q) = log sup_i P(i)/Q(i)`, the worst-case (max) log-ratio — this is why large
     `α` makes the loss increasingly sensitive to the single worst-matched bin.

**Why this matters for training.** Because `D_α` is monotonic, varying `α` changes the *relative
sensitivity* of the divergence to regions where `P` and `Q` disagree — smaller `α` weighs
under- and over-estimated regions more evenly, larger `α` weighs the worst-matched bin more
heavily. The common shorthand for this ("small α = mass-covering, large α = mode-seeking") is a
useful intuition carried over from the variational-inference literature, but it is *not* a
guarantee about what visual effect a given `α` will have on image restoration specifically —
that is an empirical question, which is exactly what Sections 9 and 15 investigate rather than
assume.

In [ ]:
def kl_divergence(p, q, eps=1e-8):
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    return (p * (p.log() - q.log())).sum(dim=(-2, -1)).mean()


def renyi_divergence(p, q, alpha=cfg.renyi_alpha, eps=1e-8):
    """Renyi divergence of order alpha. alpha=1 falls back to KL divergence exactly (Property 2)."""
    if abs(alpha - 1.0) < 1e-3:
        return kl_divergence(p, q, eps)
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    term = (p.pow(alpha) * q.pow(1.0 - alpha)).sum(dim=(-2, -1))
    term = term.clamp_min(eps)
    d_alpha = (1.0 / (alpha - 1.0)) * torch.log(term)
    return d_alpha.mean()


def _single_divergence(p, q, cfg):
    if cfg.divergence == 'kl':
        return kl_divergence(p, q)
    return renyi_divergence(p, q, alpha=cfg.renyi_alpha)


def spatial_divergence_loss(pred_img, target_img, cfg=cfg):
    """THIS is where the training loss actually reads cfg.divergence / cfg.renyi_alpha
    (see compute_loss in Section 10 for where this function itself is called).

    By default (cfg.use_shuffled_neighbor=False) this averages the divergence over MULTIPLE
    true spatial offsets (cfg.spatial_shifts) rather than a single fixed direction or a random
    pairing -- see Section 6 for why the random-pairing option breaks the spatial relationship
    entirely and is not used here."""
    if cfg.use_shuffled_neighbor:
        p_target = spatial_kde(target_img, shuffled=True)
        p_pred = spatial_kde(pred_img, shuffled=True)
        return _single_divergence(p_target, p_pred, cfg)

    divs = []
    for shift in cfg.spatial_shifts:
        p_target = spatial_kde(target_img, shift=shift, shuffled=False)
        p_pred = spatial_kde(pred_img, shift=shift, shuffled=False)
        divs.append(_single_divergence(p_target, p_pred, cfg))
    return torch.stack(divs).mean()


## 8. Sanity-Checking the Math: Monotonicity and the α→1 Limit
Before trusting `renyi_divergence` in a training loop, let's verify computationally that the
properties claimed in Section 7 actually hold for our implementation: a smooth, monotonically
increasing curve in `α`, passing exactly through the KL value at `α=1`.

In [ ]:
P_check = spatial_kde(demo_patch_high.unsqueeze(0))
Q_check = spatial_kde(demo_patch_low.unsqueeze(0))   # a genuinely different distribution

alphas_fine = np.concatenate([np.linspace(0.05, 0.99, 40), np.linspace(1.01, 5.0, 60)])
curve_vals = [renyi_divergence(P_check, Q_check, alpha=float(a)).item() for a in alphas_fine]
kl_val_check = kl_divergence(P_check, Q_check).item()

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(alphas_fine, curve_vals, color='tab:blue', label=r'$D_\alpha(P\|Q)$ (this implementation)')
ax.axhline(kl_val_check, color='tab:red', linestyle='--', label=f'$D_{{KL}}$ = {kl_val_check:.3f}')
ax.axvline(1.0, color='gray', linestyle=':', alpha=0.7)
ax.scatter([1.0], [kl_val_check], color='tab:red', zorder=5)
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel('divergence value')
ax.set_title(r'Renyi divergence is monotonic in $\alpha$ and meets KL exactly at $\alpha=1$')
ax.legend()
plt.tight_layout()
plt.show()

is_monotonic = all(b >= a - 1e-6 for a, b in zip(curve_vals, curve_vals[1:]))
print(f"Monotonic in alpha: {is_monotonic}")
print(f"D_alpha(alpha=1.0) matches D_KL to 1e-4: "
      f"{abs(renyi_divergence(P_check, Q_check, alpha=1.0).item() - kl_val_check) < 1e-4}")


## 9. Comparing KL vs. Rényi Divergence Across Real Images
A single-pair comparison can be a lucky/unlucky anecdote. Here we compute KL and Rényi
(`α ∈ {0.5, 0.75, 0.9, 0.95, 1, 1.1, 1.25, 1.5, 2, 3}`) between each real image's own spatial
distribution and its synthetically-darkened low-light counterpart, **across every base image in
the dataset**, and report mean ± std — a small but genuine aggregate comparison rather than one
example.

In [ ]:
def collect_divergence_stats(dataset, n_samples, alphas):
    """For n_samples (low, high) pairs, compute KL and Renyi(alpha) for each alpha.
    Returns a dict: {'kl': [...], 0.95: [...], 1.5: [...], ...}"""
    results = {'kl': []}
    for a in alphas:
        results[a] = []

    idxs = np.random.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
    for i in idxs:
        low_i, high_i = dataset[int(i)]
        p_t = spatial_kde(high_i.unsqueeze(0))
        p_p = spatial_kde(low_i.unsqueeze(0))
        results['kl'].append(kl_divergence(p_t, p_p).item())
        for a in alphas:
            results[a].append(renyi_divergence(p_t, p_p, alpha=a).item())
    return results


n_compare = min(16, len(real_ds))
div_stats = collect_divergence_stats(real_ds, n_compare, ALPHA_CANDIDATES)

labels = ['KL'] + [f'a={a}' for a in ALPHA_CANDIDATES]
means = [np.mean(div_stats['kl'])] + [np.mean(div_stats[a]) for a in ALPHA_CANDIDATES]
stds = [np.std(div_stats['kl'])] + [np.std(div_stats[a]) for a in ALPHA_CANDIDATES]

print(f"Divergence values aggregated over {len(div_stats['kl'])} real image pairs:")
print(f"  KL              : {means[0]:.4f} +/- {stds[0]:.4f}")
for a, m, s in zip(ALPHA_CANDIDATES, means[1:], stds[1:]):
    print(f"  Renyi (a={a:>4}) : {m:.4f} +/- {s:.4f}")

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ['tab:gray'] + ['tab:orange'] * len(ALPHA_CANDIDATES)
ax.bar(labels, means, yerr=stds, capsize=3, color=colors)
ax.set_ylabel('divergence value (mean +/- std)')
ax.set_title(f'KL vs. Renyi divergence, aggregated over {len(div_stats["kl"])} real image pairs')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 10. Combined Training Objective
The paper *replaces* the pointwise L2 loss with the spatial entropy loss (`loss_mode='replace'`).
We also expose `loss_mode='combined'` (default) as a stability-friendly ablation switch.

In [ ]:
def compute_loss(model, diffusion, low_img, high_img, cfg=cfg):
    B = high_img.shape[0]
    t = torch.randint(0, diffusion.steps, (B,), device=high_img.device).long()
    x_t, noise = diffusion.q_sample(high_img, t)
    eps_hat = model(x_t, low_img, t)

    pixel_loss = F.mse_loss(eps_hat, noise)

    # <-- THIS is where the divergence chosen in Sections 7-9/13 actually enters training.
    # spatial_divergence_loss() (Section 7) reads cfg.divergence ('renyi' or 'kl') and cfg.renyi_alpha,
    # which Section 15's ablation sets to the empirically best config found there.
    spatial_loss = spatial_divergence_loss(eps_hat, noise, cfg)

    if cfg.loss_mode == 'replace':
        total = spatial_loss
    else:
        total = cfg.lambda_pixel * pixel_loss + cfg.lambda_spatial * spatial_loss

    return total, {'pixel_loss': pixel_loss.item(), 'spatial_loss': spatial_loss.item(),
                    'total_loss': total.item()}


## 11. Optimizer, LR Schedule, EMA — Optimization Choices
`AdamW` + cosine decay match Table 4. Added on top: mixed precision (AMP), gradient clipping
(stabilizes against occasional large KDE-loss gradients), and EMA weights for smoother inference.

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def build_optim(model, cfg):
    optim = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = CosineAnnealingLR(optim, T_max=max(cfg.total_iters, 1))
    return optim, sched


def lr_warmup(optim, step, warmup_iters, base_lr):
    if warmup_iters > 0 and step < warmup_iters:
        lr = base_lr * (step + 1) / warmup_iters
        for g in optim.param_groups:
            g['lr'] = lr


## 12. Training Loop
Also tracks the gradient norm at every step (useful diagnostic: spikes here usually mean the KDE
loss briefly dominated, and is exactly the kind of curve a research report would include).

In [ ]:
def train(model, diffusion, dataloader, cfg, log_every=50, ckpt_dir='checkpoints'):
    os.makedirs(ckpt_dir, exist_ok=True)
    model.to(DEVICE)
    optim, sched = build_optim(model, cfg)
    scaler = GradScaler(enabled=cfg.mixed_precision)
    ema = EMA(model, cfg.ema_decay)

    history = {'step': [], 'pixel_loss': [], 'spatial_loss': [], 'total_loss': [],
               'lr': [], 'grad_norm': []}

    data_iter = iter(dataloader)
    model.train()
    t0 = time.time()

    for step in range(cfg.total_iters):
        try:
            low, high = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            low, high = next(data_iter)
        low, high = low.to(DEVICE), high.to(DEVICE)

        lr_warmup(optim, step, cfg.warmup_iters, cfg.lr)
        optim.zero_grad(set_to_none=True)

        with autocast(enabled=cfg.mixed_precision):
            loss, logs = compute_loss(model, diffusion, low, high, cfg)

        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optim)
        scaler.update()
        if step >= cfg.warmup_iters:
            sched.step()

        ema.update(model)

        history['step'].append(step)
        history['pixel_loss'].append(logs['pixel_loss'])
        history['spatial_loss'].append(logs['spatial_loss'])
        history['total_loss'].append(logs['total_loss'])
        history['lr'].append(optim.param_groups[0]['lr'])
        history['grad_norm'].append(float(grad_norm))

        if step % log_every == 0:
            elapsed = time.time() - t0
            print(f"step {step:6d}/{cfg.total_iters} | "
                  f"pixel {logs['pixel_loss']:.4f} | spatial({cfg.divergence}) {logs['spatial_loss']:.4f} "
                  f"| total {logs['total_loss']:.4f} | grad_norm {float(grad_norm):.3f} "
                  f"| lr {optim.param_groups[0]['lr']:.2e} | {elapsed:.1f}s")

        if step > 0 and step % 1000 == 0:
            torch.save({'model': model.state_dict(), 'ema': ema.shadow, 'step': step},
                       os.path.join(ckpt_dir, f'ckpt_{step}.pt'))

    return model, ema, history


## 13. Sampling / Inference
DDPM reverse process conditioned on the low-light input at every step. EMA weights are typically
used at inference time for smoother samples.

In [ ]:
@torch.no_grad()
def enhance(model, diffusion, low_img, device=DEVICE):
    model.eval()
    low_img = low_img.to(device)
    shape = low_img.shape
    out = diffusion.p_sample_loop(model, low_img, shape, device)
    return out


## 14. Evaluation — PSNR / SSIM
Standard distortion metrics reported in the paper's tables (LPIPS/FID need extra pretrained
networks and are omitted for simplicity).

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def to_numpy_img(t):
    img = (t.clamp(-1, 1) + 1) / 2
    img = img.permute(1, 2, 0).cpu().numpy()
    return img

def evaluate_pair(pred, target):
    p_np, t_np = to_numpy_img(pred), to_numpy_img(target)
    p_score = psnr(t_np, p_np, data_range=1.0)
    s_score = ssim(t_np, p_np, data_range=1.0, channel_axis=-1)
    return p_score, s_score


## 15. Selecting the Most Suitable α (Staged Ablation Study)
Section 9 compared divergences as a static *measurement*; here we compare them as an actual
**training signal**, and — importantly — score them by **validation PSNR/SSIM**, not raw training
loss. This matters because `D_α` is monotonic in `α` (Section 7): a Rényi loss at a small `α` is
*numerically* smaller than one at a large `α` almost by construction, so comparing raw loss values
across different `α` mostly measures the divergence's own scale, not which model actually restores
images better. Validation PSNR/SSIM are computed the same way regardless of which divergence
trained the model, so they're the fair comparison metric.

**Two-stage design** (screening many candidates cheaply, then confirming the best ones properly):

1. **Screening** — train every candidate (`KL` baseline + 10 `Rényi α` values spanning
   `{0.5, ..., 3.0}`) for a short, identical budget, then evaluate each on the **validation split**
   (never trained on — see Section 3). This is cheap enough to cover a wide `α` range.
2. **Confirmation** — take the top-2 screening candidates plus the `KL` baseline (3 configs total)
   and train them for a longer budget, then re-evaluate on validation. The winner of *this* stage
   — not the screening stage — is what Section 17's full run actually uses, since a longer budget
   is more representative of real training dynamics than the short screen.

**Note on scope:** even the confirmation stage is single-seed and short relative to the paper's
`250_000`-iteration setting — see Section 22 for this limitation stated explicitly, along with
what a fully rigorous version of this ablation would add (multi-seed, longer confirmation runs).

In [ ]:
def make_fresh_model_and_diffusion():
    torch.manual_seed(SEED)   # identical init across configs for a fair comparison
    m = ConditionalNAFNet(base_ch=cfg.base_channels, mults=cfg.channel_mults,
                           time_emb_dim=cfg.time_emb_dim).to(DEVICE)
    d = GaussianDiffusion(cfg.diffusion_steps, cfg.beta_schedule, device=DEVICE)
    return m, d


def run_short_training(divergence, alpha, iters, patch_size=64, batch_size=8, log_every=9999):
    """Train briefly under a fixed divergence choice on the TRAIN split; return history + model."""
    local_cfg = Config()
    local_cfg.divergence = divergence
    local_cfg.renyi_alpha = alpha
    local_cfg.total_iters = iters
    local_cfg.patch_size = patch_size
    local_cfg.batch_size = batch_size
    local_cfg.warmup_iters = min(20, iters // 4)

    if USING_LOL_V1:
        ds = LOLDataset(_low_dir, _high_dir, patch_size=patch_size, train=True,
                         indices=list(range(0, len(glob.glob(os.path.join(_low_dir, '*.*'))) - VAL_SIZE)))
    else:
        ds = RealSampleDataset(patch_size=patch_size, length=128, train=True, split='train')
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)

    model, diffusion = make_fresh_model_and_diffusion()
    _, _, history = train(model, diffusion, loader, local_cfg, log_every=log_every,
                           ckpt_dir=f'ckpt_ablation_{divergence}_{alpha}')
    return history, model, diffusion


@torch.no_grad()
def evaluate_on_validation(model, diffusion, val_dataset, n_val, patch_size):
    """Mean PSNR/SSIM on up to n_val samples from a held-out validation set (never trained on)."""
    n = min(n_val, len(val_dataset))
    psnrs, ssims = [], []
    for i in range(n):
        low_i, high_i = val_dataset[i]
        pred_i = enhance(model, diffusion, low_i.unsqueeze(0))[0].cpu()
        p, s = evaluate_pair(pred_i, high_i)
        psnrs.append(p)
        ssims.append(s)
    return float(np.mean(psnrs)), float(np.mean(ssims))


N_VAL_ABLATION = 6          # validation samples scored per ablation config (keep small for speed)
SCREEN_ITERS = 300          # short screening budget per candidate
CONFIRM_ITERS = 1200        # longer confirmation budget for the top-2 + KL
ABLATION_PATCH = 64         # smaller than the paper's 128, purely to keep the ablation affordable

screen_configs = [('kl', 1.0, 'KL (baseline)')]
for a in ALPHA_CANDIDATES:
    screen_configs.append(('renyi', a, f'Renyi a={a}'))


In [ ]:
# --- Stage 1: screening ---
screen_histories, screen_val_psnr, screen_val_ssim = {}, {}, {}
t0 = time.time()
for divergence, alpha, label in screen_configs:
    print(f'--- [screen] {label} ---')
    hist, model_s, diffusion_s = run_short_training(divergence, alpha, SCREEN_ITERS, patch_size=ABLATION_PATCH)
    p, s = evaluate_on_validation(model_s, diffusion_s, real_val_ds, N_VAL_ABLATION, ABLATION_PATCH)
    screen_histories[label] = hist
    screen_val_psnr[label] = p
    screen_val_ssim[label] = s
    print(f'    val PSNR={p:.2f} dB, val SSIM={s:.4f}')
print(f'Screening finished in {time.time() - t0:.1f}s total.')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
labels_ord = list(screen_val_psnr.keys())
vals_ord = [screen_val_psnr[l] for l in labels_ord]
colors = ['tab:gray' if l.startswith('KL') else 'tab:orange' for l in labels_ord]
ax.bar(labels_ord, vals_ord, color=colors)
ax.set_ylabel('validation PSNR (dB)')
ax.set_title(f'Screening stage ({SCREEN_ITERS} iters/config): validation PSNR per config')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Top-2 by validation PSNR, always including KL for reference in the confirmation stage
ranked = sorted(screen_val_psnr, key=screen_val_psnr.get, reverse=True)
top2 = [l for l in ranked if not l.startswith('KL')][:2]
confirm_labels = list(dict.fromkeys(['KL (baseline)'] + top2))   # dedupe, keep order
print(f'Screening ranking (best first): {ranked}')
print(f'Advancing to confirmation stage: {confirm_labels}')


In [ ]:
# --- Stage 2: confirmation (longer budget, top-2 + KL) ---
def label_to_config(label):
    if label.startswith('KL'):
        return 'kl', 1.0
    return 'renyi', float(label.split('a=')[1])

confirm_histories, confirm_val_psnr, confirm_val_ssim = {}, {}, {}
t0 = time.time()
for label in confirm_labels:
    divergence, alpha = label_to_config(label)
    print(f'--- [confirm] {label} ---')
    hist, model_c, diffusion_c = run_short_training(divergence, alpha, CONFIRM_ITERS, patch_size=ABLATION_PATCH)
    p, s = evaluate_on_validation(model_c, diffusion_c, real_val_ds, N_VAL_ABLATION, ABLATION_PATCH)
    confirm_histories[label] = hist
    confirm_val_psnr[label] = p
    confirm_val_ssim[label] = s
    print(f'    val PSNR={p:.2f} dB, val SSIM={s:.4f}')
print(f'Confirmation finished in {time.time() - t0:.1f}s total.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for label, hist in confirm_histories.items():
    axes[0].plot(hist['step'], hist['total_loss'], label=label, alpha=0.85)
axes[0].set_xlabel('step'); axes[0].set_ylabel('total loss')
axes[0].set_title(f'Confirmation stage ({CONFIRM_ITERS} iters): training loss')
axes[0].legend(fontsize=8)

labels_c = list(confirm_val_psnr.keys())
vals_c = [confirm_val_psnr[l] for l in labels_c]
colors_c = ['tab:gray' if l.startswith('KL') else 'tab:orange' for l in labels_c]
axes[1].bar(labels_c, vals_c, color=colors_c)
axes[1].set_ylabel('validation PSNR (dB)')
axes[1].set_title('Confirmation stage: validation PSNR (this decides the winner)')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

best_label = max(confirm_val_psnr, key=confirm_val_psnr.get)
print(f"Winner (by validation PSNR after confirmation stage): {best_label} "
      f"(val PSNR={confirm_val_psnr[best_label]:.2f} dB, val SSIM={confirm_val_ssim[best_label]:.4f})")

cfg.divergence, cfg.renyi_alpha = label_to_config(best_label)
print(f"--> cfg.divergence = '{cfg.divergence}', cfg.renyi_alpha = {cfg.renyi_alpha} "
      f"(this is what Section 17's full training run will use)")


## 16. Benchmarking KDE Cost Before Committing to a Full Run
The spatial KDE's `einsum('bcni,bcnj->bcij', ...)` scales with `pixels × bins²`, and Section 6
now averages over 4 neighbor offsets instead of 1 — a real 4x cost increase on the spatial-loss
portion of every step. Before launching a `total_iters`-scale run, it's worth measuring how much
of each step's wall-clock time is actually the KDE/divergence computation vs. the model's own
forward/backward pass, so you know whether it's worth optimizing (fewer bins, fewer offsets,
larger bandwidth) before spending compute at scale.

In [ ]:
def benchmark_step_cost(model, diffusion, patch_size, batch_size, n_repeats=5):
    model.to(DEVICE)
    low = torch.randn(batch_size, 3, patch_size, patch_size, device=DEVICE)
    high = torch.randn(batch_size, 3, patch_size, patch_size, device=DEVICE)

    # warm-up (compilation / cudnn autotune / first-call overhead shouldn't count)
    for _ in range(2):
        compute_loss(model, diffusion, low, high, cfg)

    t0 = time.time()
    for _ in range(n_repeats):
        loss, _ = compute_loss(model, diffusion, low, high, cfg)
        loss.backward()
    total_time = (time.time() - t0) / n_repeats

    # isolate just the KDE+divergence portion for comparison
    t0 = time.time()
    for _ in range(n_repeats):
        spatial_divergence_loss(high, low, cfg)
    kde_time = (time.time() - t0) / n_repeats

    return total_time, kde_time


bench_model = ConditionalNAFNet(base_ch=cfg.base_channels, mults=cfg.channel_mults,
                                 time_emb_dim=cfg.time_emb_dim)
bench_diffusion = GaussianDiffusion(cfg.diffusion_steps, cfg.beta_schedule, device=DEVICE)
full_time, kde_time = benchmark_step_cost(bench_model, bench_diffusion, patch_size=128, batch_size=cfg.batch_size)

print(f"Avg full step time (fwd+bwd): {full_time*1000:.1f} ms")
print(f"Avg spatial-divergence-only time: {kde_time*1000:.1f} ms "
      f"({100*kde_time/full_time:.0f}% of the full step)")
del bench_model, bench_diffusion


## 17. Full Training Run
Uses the **single alpha chosen by Section 15's confirmation stage** (printed above as
`cfg.renyi_alpha`), the real dataset resolved in Section 3 (training split only — validation and
test stay untouched here), and `patch_size=128` as in the paper. The main training run uses
`total_iters=250_000`.

In [ ]:
cfg.patch_size = 128                 # paper's setting, restored from the ablation's smaller patch
cfg.total_iters = 250_000             # main training: 250,000 iterations
cfg.batch_size = 8 if DEVICE.type == 'cpu' else 16   # paper: 16 (use 16 on GPU)

print(f"Estimated wall-clock for this run at the benchmarked per-step cost: "
      f"~{full_time * cfg.total_iters / 3600:.1f} hours ({full_time*1000:.0f} ms/step x "
      f"{cfg.total_iters} steps) -- rough estimate only, actual GPU throughput will differ from CPU.")

if USING_LOL_V1:
    train_ds = LOLDataset(_low_dir, _high_dir, patch_size=cfg.patch_size, train=True,
                           indices=list(range(0, len(glob.glob(os.path.join(_low_dir, '*.*'))) - VAL_SIZE)))
else:
    train_ds = RealSampleDataset(patch_size=cfg.patch_size, length=256, train=True, split='train')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=0, drop_last=True)

print(f"Training on: {DATASET_SOURCE}, patch_size={cfg.patch_size}, "
      f"total_iters={cfg.total_iters}, divergence={cfg.divergence}, alpha={cfg.renyi_alpha}")

model = ConditionalNAFNet(base_ch=cfg.base_channels, mults=cfg.channel_mults,
                           time_emb_dim=cfg.time_emb_dim)
diffusion = GaussianDiffusion(cfg.diffusion_steps, cfg.beta_schedule, device=DEVICE)

model, ema, history = train(model, diffusion, train_loader, cfg, log_every=200)


## 18. Training Curves
Pixel loss, spatial divergence loss, total loss, gradient norm, and LR schedule over the full
training run. On real data these should trend downward as the model learns.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

axes[0, 0].plot(history['step'], history['pixel_loss'], color='tab:blue')
axes[0, 0].set_title('Pixel (L2 noise-matching) loss')
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('loss')

axes[0, 1].plot(history['step'], history['spatial_loss'], color='tab:orange')
axes[0, 1].set_title(f"Spatial divergence loss ({cfg.divergence}, alpha={cfg.renyi_alpha})")
axes[0, 1].set_xlabel('step'); axes[0, 1].set_ylabel('loss')

axes[0, 2].plot(history['step'], history['total_loss'], color='tab:green')
axes[0, 2].set_title('Total combined loss')
axes[0, 2].set_xlabel('step'); axes[0, 2].set_ylabel('loss')

axes[1, 0].plot(history['step'], history['grad_norm'], color='tab:purple', alpha=0.8)
axes[1, 0].set_title('Gradient norm (post-clip)')
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylabel('norm')

axes[1, 1].plot(history['step'], history['lr'], color='tab:red')
axes[1, 1].set_title('Learning rate schedule')
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylabel('lr')

# Smoothed total loss (moving average) makes the trend easier to read through the noise
window = max(1, len(history['total_loss']) // 50)
if len(history['total_loss']) >= window > 1:
    smoothed = np.convolve(history['total_loss'], np.ones(window) / window, mode='valid')
    axes[1, 2].plot(history['step'][:len(smoothed)], smoothed, color='tab:green')
    axes[1, 2].set_title(f'Total loss (moving avg, window={window})')
else:
    axes[1, 2].plot(history['step'], history['total_loss'], color='tab:green')
    axes[1, 2].set_title('Total loss')
axes[1, 2].set_xlabel('step'); axes[1, 2].set_ylabel('loss')

plt.tight_layout()
plt.show()


## 19. Spatial Distribution Heatmaps: Target vs. Prediction, Before vs. After Training
The most direct visual evidence of what the spatial-entropy loss is optimizing: the model's
predicted-noise spatial distribution `P_pred` should move closer to the true-noise distribution
`P_target` as training progresses. We compare a freshly-initialized model against the trained one
on the same real validation patch.

In [ ]:
@torch.no_grad()
def get_noise_distributions(model, diffusion, low_img, high_img, t_val=50):
    t = torch.full((1,), t_val, device=DEVICE, dtype=torch.long)
    x_t, noise = diffusion.q_sample(high_img.unsqueeze(0).to(DEVICE), t)
    eps_hat = model(x_t, low_img.unsqueeze(0).to(DEVICE), t)
    p_target = spatial_kde(noise)[0, 0].cpu().numpy()
    p_pred = spatial_kde(eps_hat)[0, 0].cpu().numpy()
    return p_target, p_pred

val_low, val_high = train_ds[0]

fresh_model, fresh_diffusion = make_fresh_model_and_diffusion()
p_target_fresh, p_pred_fresh = get_noise_distributions(fresh_model, fresh_diffusion, val_low, val_high)
p_target_trained, p_pred_trained = get_noise_distributions(model, diffusion, val_low, val_high)

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, data, title in zip(
    axes.ravel(),
    [p_target_fresh, p_pred_fresh, p_target_trained, p_pred_trained],
    ['P_target (untrained model)', 'P_pred (untrained model)',
     'P_target (trained model)', 'P_pred (trained model)']):
    im = ax.imshow(data, origin='lower', cmap='inferno')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Spatial KDE distributions: target vs. predicted noise, before vs. after training')
plt.tight_layout()
plt.show()

kl_before = kl_divergence(torch.from_numpy(p_target_fresh).unsqueeze(0).unsqueeze(0),
                           torch.from_numpy(p_pred_fresh).unsqueeze(0).unsqueeze(0)).item()
kl_after = kl_divergence(torch.from_numpy(p_target_trained).unsqueeze(0).unsqueeze(0),
                          torch.from_numpy(p_pred_trained).unsqueeze(0).unsqueeze(0)).item()
print(f"KL(P_target, P_pred) before training: {kl_before:.4f}")
print(f"KL(P_target, P_pred) after training : {kl_after:.4f}")


## 20. Qualitative Results — Low / Enhanced / Ground-Truth
Side-by-side comparison for a few validation samples from the real dataset in use.

In [ ]:
def show_triplet(low, pred, high, idx=0):
    imgs = [to_numpy_img(low), to_numpy_img(pred), to_numpy_img(high)]
    titles = ['Low-light input', 'Enhanced (model output)', 'Ground truth']
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.suptitle(f'Sample {idx}')
    plt.tight_layout()
    plt.show()

n_show = 3
for i in range(n_show):
    low_i, high_i = train_ds[i]
    pred_i = enhance(model, diffusion, low_i.unsqueeze(0))[0].cpu()
    show_triplet(low_i, pred_i, high_i, idx=i)


## 21. Quantitative Metrics Summary — Final Test-Set Evaluation
This is the **one and only** place the held-out **test split** (Section 3) is used — the model
never trained on it, and it wasn't used to pick `α` either (that was validation, in Section 15).
This is the fair, final number for the chosen config, plus a comparison back to the KL baseline
from the confirmation stage so the Rényi-vs-KL comparison ends in an honest, non-circular result.

In [ ]:
n_eval = min(8, len(real_test_ds))
psnr_scores, ssim_scores = [], []

for i in range(n_eval):
    low_i, high_i = real_test_ds[i % len(real_test_ds)]
    pred_i = enhance(model, diffusion, low_i.unsqueeze(0))[0].cpu()
    p, s = evaluate_pair(pred_i, high_i)
    psnr_scores.append(p)
    ssim_scores.append(s)

psnr_arr, ssim_arr = np.array(psnr_scores), np.array(ssim_scores)
print(f"Final model ({cfg.divergence}, alpha={cfg.renyi_alpha}) on TEST split -- "
      f"PSNR: {psnr_arr.mean():.2f} +/- {psnr_arr.std():.2f} dB  (n={n_eval})")
print(f"Final model ({cfg.divergence}, alpha={cfg.renyi_alpha}) on TEST split -- "
      f"SSIM: {ssim_arr.mean():.4f} +/- {ssim_arr.std():.4f}  (n={n_eval})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(n_eval), psnr_scores, color='tab:blue')
axes[0].axhline(psnr_arr.mean(), color='k', linestyle='--', linewidth=1, label='mean')
axes[0].set_title('Test-set PSNR per sample (dB)'); axes[0].set_xlabel('sample idx'); axes[0].legend()

axes[1].bar(range(n_eval), ssim_scores, color='tab:orange')
axes[1].axhline(ssim_arr.mean(), color='k', linestyle='--', linewidth=1, label='mean')
axes[1].set_title('Test-set SSIM per sample'); axes[1].set_xlabel('sample idx'); axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Results table: Section 15's screening + confirmation validation PSNR per config, alongside
# what actually got selected -- ties the measurement-level comparison to a concrete outcome.
print(f"{'Config':<18}{'Screen val PSNR':<18}{'Confirm val PSNR':<20}{'Selected?':<12}")
print('-' * 68)
for label in screen_val_psnr:
    confirm_str = f"{confirm_val_psnr[label]:.2f}" if label in confirm_val_psnr else '--'
    is_selected = (label == best_label)
    print(f"{label:<18}{screen_val_psnr[label]:<18.2f}{confirm_str:<20}{'<-- YES' if is_selected else '':<12}")


## 22. Discussion, Limitations, and Reproduction Notes

**Summary of findings in this run.** Section 8 confirmed computationally that our Rényi
implementation is monotonic in `α` and exactly reproduces KL at `α=1`, matching the theory in
Section 7. Section 9 showed the same ordering holds on real image statistics, not just synthetic
distributions. Section 15's two-stage ablation used **validation PSNR/SSIM** — not raw training
loss — as the selection criterion (raw loss isn't comparable across `α` since `D_α` is monotonic
in `α` by construction, so a smaller loss value can just mean a smaller-`α` divergence, not a
better model) and selected a single `α` (printed above) for the full run.

**Issues from an earlier version of this notebook that are now fixed, not just noted:**
- *Shuffled neighbor pairing.* An earlier version set `use_shuffled_neighbor=True` by default,
  which pairs each pixel with a uniformly random pixel from the same image rather than an actual
  spatial neighbor — silently breaking the "spatial" in spatial KDE. Section 6 now defaults to
  true spatial neighbors, averaged over 4 offsets for robustness to edge orientation.
- *Loss-based α selection.* An earlier version picked `α` by comparing raw training loss values
  across configs, which is invalid given `D_α`'s monotonicity (see above). Section 15 now selects
  by validation-set PSNR, which is on the same scale regardless of which divergence trained the
  model.
- *Train/val/test leakage.* An earlier version evaluated on the same images used for training.
  Section 3 now carves out a real validation split (never trained on, used only to pick `α`) and a
  test split (touched exactly once, in Section 21, after `α` is already fixed).

**Limitations that remain (stated plainly, as a paper would):**
- *Single seed.* Both the screening and confirmation stages in Section 15 use one training seed.
  A rigorous ablation would repeat each config over multiple seeds and report confidence intervals
  on validation PSNR/SSIM, not a point estimate.
- *Confirmation budget still short.* `CONFIRM_ITERS=1200` is enough to separate the top candidates
  from clearly-worse ones on this small dataset, but is nowhere near the paper's `250_000`
  iterations — the ranking could still shift at full convergence.
- *Dataset scale.* Depending on what downloaded successfully in Section 3, results here reflect
  either LOL-v1's real (but small) splits or a 4-image synthetic-real fallback (where validation
  and test are forced to share their one held-out image) — both far short of the diversity needed
  for results to generalize the way the original paper's full LOL-v1/v2 benchmarks would.
- *No LPIPS/FID.* We report PSNR/SSIM only; the original paper also reports perceptual metrics
  (LPIPS, FID) which need extra pretrained networks not included here, and which the paper's own
  motivation (moving past pixel-wise L2) is specifically about.
- *Compute budget for the main run.* `total_iters=250_000` at `patch_size=128` matches the
  paper's stated iteration budget; wall-clock time still depends on the hardware and KDE cost.
- *KDE cost.* Averaging over 4 spatial offsets (Section 6's fix) is a real ~4x cost increase on
  the spatial-loss term versus a single offset; Section 16's benchmark cell quantifies this so the
  tradeoff against fewer offsets/bins is an informed choice, not a guess.

**What would strengthen this into a publishable ablation:**
1. Repeat Section 15's confirmation stage over ≥3 seeds per candidate, reporting mean±std
   validation PSNR/SSIM with confidence intervals, not point estimates.
2. Run the full `250_000`-iteration training separately for the confirmation stage's top candidate
   and the KL baseline to verify the short-run ranking holds at convergence.
3. Add LPIPS/FID for perceptual comparison alongside PSNR/SSIM.
4. Extend the `α` grid further, or use a proper 1-D optimization (e.g. golden-section search) over
   `α` instead of a fixed candidate list, to more precisely locate any optimum away from `α=1`.

**Reproducing the paper exactly:** `cfg.patch_size=128`, `cfg.batch_size=16`, `cfg.lr=1e-4`,
`cfg.total_iters=250_000`, `cfg.diffusion_steps=100`, AdamW + cosine schedule, and real LOL-v1/v2
data (Section 3 downloads LOL-v1 automatically; LOL-v2 needs manual download per Section 3's
notes). Expect roughly 1–2 days on a single A100 for the full 250k iterations (a rough planning
estimate, not an author-reported number — actual time depends heavily on the KDE implementation,
mixed-precision usage, and data pipeline).


## 23. References

1. Lian, W., Lian, W., Luo, Z. *Equipping Diffusion Models with Differentiable Spatial Entropy for
   Low-Light Image Enhancement.* CVPR Workshops (NTIRE) 2024. arXiv:2404.09735. Official code:
   `github.com/shermanlian/spatial-entropy-loss`.
2. Van Erven, T., Harremoës, P. *Rényi Divergence and Kullback–Leibler Divergence.* IEEE
   Transactions on Information Theory, 60(7), 2014 — source for the monotonicity, limit, and
   special-case properties used in Section 7.
3. Wei, C., Wang, W., Yang, W., Liu, J. *Deep Retinex Decomposition for Low-Light Enhancement.*
   BMVC 2018 — introduces the LOL-v1 dataset used in Section 3.
4. Yang, W. et al. *Sparse Gradient Regularized Deep Retinex Network for Robust Low-Light Image
   Enhancement.* IEEE TIP, 2021 — introduces LOL-v2 (real & synthetic).
5. Chen, L. et al. *Simple Baselines for Image Restoration (NAFNet).* ECCV 2022 — architecture
   basis for the conditional noise predictor in Section 4.
6. Ho, J., Jain, A., Abbeel, P. *Denoising Diffusion Probabilistic Models.* NeurIPS 2020 — the
   diffusion formulation used in Section 5.
7. LOL-v1 dataset mirror used in Section 3's automatic download:
   `huggingface.co/datasets/geekyrakshit/LoL-Dataset` (as used by the official Keras Zero-DCE
   tutorial, `keras.io/examples/vision/zero_dce`).
